[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/exercises/basic/03_ml_workflow.ipynb)

# Luyện tập cơ bản 3 — Quy trình ML với scikit-learn

Chia dữ liệu, pipeline, baseline, metric, cross-validation, đường cong học, lưu mô hình — các thao tác lặp lại trong mọi dự án.

Chương liên quan: 04, 05, 06, 07, 14, 15.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/exercises/basic

In [ ]:
import tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import confusion_matrix, mean_absolute_error, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, learning_curve, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

churn = pd.read_csv(repo_path("data", "sample", "customer_churn.csv"))
houses = pd.read_csv(repo_path("data", "sample", "house_prices_dirty.csv")).drop_duplicates()

**Bài 1.** Chia `churn` 80/20 **có phân tầng** theo `churn` (`random_state=0`) → `train`, `test`. Tỉ lệ rời bỏ của hai phần phải gần như bằng nhau.

In [ ]:
### START CODE HERE ### (≈ 1 dòng)
train, test = None, None
### END CODE HERE ###
print(f"{train.churn.mean():.4f} vs {test.churn.mean():.4f}")

In [ ]:
check_equal(len(test), 1000, "20% làm test")
check_less(abs(train.churn.mean() - test.churn.mean()), 0.002, "phân tầng giữ tỉ lệ lớp")

**Bài 2.** Với `houses` (chỉ dùng các cột số `NUMERIC`, mục tiêu `price_billion_vnd`; chia 80/20 `random_state=0`): pipeline `SimpleImputer(median) → StandardScaler → LinearRegression` → `mae_model`; baseline `DummyRegressor(strategy="median")` → `mae_baseline`.

In [ ]:
NUMERIC = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
h_tr, h_te = train_test_split(houses, test_size=0.2, random_state=0)
### START CODE HERE ### (≈ 4 dòng)
model = None
mae_model = None
baseline = None
mae_baseline = None
### END CODE HERE ###
print(f"MAE mô hình {mae_model:.3f} vs baseline {mae_baseline:.3f}")

In [ ]:
check_less(mae_model, mae_baseline, "mô hình tốt hơn baseline")
check_true(h_te[NUMERIC].isna().any().any(), "test có giá trị thiếu — pipeline tự xử lý")

**Bài 3.** Pipeline đầy đủ cho `churn`: chuẩn hoá cột số, one-hot cột phân loại, `LogisticRegression(max_iter=1000)` → `clf` (fit trên `train`); dự đoán nhãn test `pred`; tính **từ ma trận nhầm lẫn** `precision_manual`, `recall_manual`.

In [ ]:
NUM_C = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products"]
CAT_C = ["contract", "internet_service", "payment_method", "senior"]
### START CODE HERE ### (≈ 6 dòng)
clf = None
pred = None
tn, fp, fn, tp = None, None, None, None
precision_manual = None
recall_manual = None
### END CODE HERE ###
print(f"precision {precision_manual:.3f}, recall {recall_manual:.3f}")

In [ ]:
check_close(precision_manual, precision_score(test.churn, pred), message="precision")
check_close(recall_manual, recall_score(test.churn, pred), message="recall")

**Bài 4.** 5-fold CV phân tầng (`shuffle=True, random_state=0`) ROC-AUC của `clf` trên `train` → `cv_scores`; báo cáo trung bình ± độ lệch chuẩn.

In [ ]:
### START CODE HERE ### (≈ 1 dòng)
cv_scores = None
### END CODE HERE ###
print(f"ROC-AUC CV = {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

In [ ]:
check_equal(len(cv_scores), 5, "5 fold")
check_greater(cv_scores.mean(), 0.8, "AUC CV")
check_less(cv_scores.std(), 0.03, "ổn định giữa các fold")

**Bài 5.** Đường cong học: dùng `learning_curve` với `DecisionTreeClassifier(random_state=0)` (không giới hạn độ sâu) trên các cột số của train, `train_sizes=[0.1, 0.3, 0.6, 1.0]`, 5-fold, ROC-AUC → `gap` = AUC train trung bình − AUC validation trung bình ở kích thước lớn nhất. Cây sâu đang overfit hay underfit?

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
sizes, tr_scores, va_scores = None, None, None
gap = None
### END CODE HERE ###
print(f"AUC train {tr_scores[-1].mean():.3f} | validation {va_scores[-1].mean():.3f} | chênh {gap:.3f}")

In [ ]:
check_greater(gap, 0.2, "chênh lệch lớn → overfit (cây thuộc lòng train)")

**Bài 6.** Lưu `clf` bằng `joblib` vào thư mục tạm, nạp lại → `reloaded`; dự đoán xác suất phải khớp.

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
path = None
reloaded = None
### END CODE HERE ###

In [ ]:
check_close(reloaded.predict_proba(test)[:, 1], clf.predict_proba(test)[:, 1], message="nạp lại cho cùng dự đoán")